In [2]:
from google.colab import drive
drive.mount('/content/gdrive')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import gaussian, threshold_otsu
from skimage.segmentation import *
from skimage.color import rgb2gray, rgb2hsv
import skimage.io as skio
import re
from pathlib import Path
from typing import Literal
from skimage.feature import graycomatrix, graycoprops
import skimage.morphology as skmor
import skimage.measure as skmea
from torch.utils.data import Dataset
from skimage.morphology import (
    disk,
    binary_opening,
    binary_closing,
    remove_small_objects,
    remove_small_holes
)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
#!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img"
#!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img_large.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img_large"

chatgpt5.4 is used for designing and debugging structure of this dataset class.

In [ ]:
class Hep2Dataset(Dataset):
    def __init__(self, dataset_root, csv_path, transform=None):
        """
        Initialize the dataset.

        Args:
            dataset_root (str or Path): Root folder of the image dataset.
            csv_path (str or Path): CSV file containing image file names and labels.
            transform (callable, optional): Transform to apply to each image.
        """
        self.dataset_root = Path(dataset_root)
        self.csv_path = Path(csv_path)
        self.transform = transform

        # Read all samples once and store them as a unified list
        self.samples = self.read_label(self.csv_path)

    def read_label(self, csv_path):
        """
        Read the CSV file and build a list of (image_path, label).

        Args:
            csv_path (Path): Path to the CSV label file.

        Returns:
            list: A list of tuples, where each tuple is (image_path, label).
        """
        df = pd.read_csv(csv_path)

        self.samples = []
        for _, row in df.iterrows():
            img_name = row["file"]
            label = int(row["class"])  # Convert label to integer
            mask_path = self.dataset_root / f'{img_name}_Mask.png'
            img_path = self.dataset_root / f'{img_name}.png'
            self.samples.append((img_path, mask_path, label))

        return self.samples

    def __len__(self):
        """
        Return the total number of samples in the dataset.
        """
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Get one sample by index.

        Args:
            idx (int): Index of the sample.

        Returns:
            tuple: (image, label)
        """
        img_path, mask_path, label = self.samples[idx]
        img = skio.imread(img_path)
        mask = skio.imread(mask_path)

        if self.transform is not None:
            img = self.transform(img)

        return img, label, mask

chatgpt5.4 used for designing shape feature extraction class.
Here I used circularity, solidity and eccentricity beacuse we don't have metadata like what we have in DICOM format. So some features like perimeter and area, we cannnot get.

In [ ]:
import numpy as np
import skimage.measure as skmea


class shape_feature_exatract:
    def __init__(self, images, labels, masks):
        """
        Initialize the shape feature extractor.

        Args:
            images (list or ndarray): Input images.
            labels (list or ndarray): Corresponding labels.
            masks (list or ndarray): Binary or grayscale masks.
        """
        self.image = images
        self.label = labels
        self.masks = masks

        self.circularity = []
        self.solidity = []
        self.eccentricity = []

        self.shape_feature = {}

    def _get_largest_region(self, mask):
        """
        Get the largest connected component from a mask.

        Args:
            mask (ndarray): Input mask.

        Returns:
            regionprops object or None: Largest region if exists, else None.
        """
        mask = mask > 0
        labeled_mask = skmea.label(mask)

        regions = skmea.regionprops(labeled_mask)
        if len(regions) == 0:
            return None

        largest_region = max(regions, key=lambda x: x.area)
        return largest_region

    def get_solidity(self, mask):
        """
        Calculate solidity of the largest connected component.

        Args:
            mask (ndarray): Input mask.

        Returns:
            float: Solidity value. Returns 0.0 if no valid region exists.
        """
        largest_region = self._get_largest_region(mask)
        if largest_region is None:
            return 0.0

        solidity = largest_region.solidity
        return float(solidity)

    def get_eccentricity(self, mask):
        """
        Calculate eccentricity of the largest connected component.

        Args:
            mask (ndarray): Input mask.

        Returns:
            float: Eccentricity value. Returns 0.0 if no valid region exists.
        """
        largest_region = self._get_largest_region(mask)
        if largest_region is None:
            return 0.0

        eccentricity = largest_region.eccentricity
        return float(eccentricity)

    def get_circularity(self, mask):
        """
        Calculate circularity of the largest connected component.

        Formula:
            circularity = 4 * pi * area / perimeter^2

        Args:
            mask (ndarray): Input mask.

        Returns:
            float: Circularity value. Returns 0.0 if no valid region exists
                   or perimeter is zero.
        """
        largest_region = self._get_largest_region(mask)
        if largest_region is None:
            return 0.0

        area = largest_region.area
        perimeter = largest_region.perimeter

        if perimeter == 0:
            return 0.0

        circularity = 4.0 * np.pi * area / (perimeter ** 2)
        return float(circularity)

    def extract_single_shape_feature(self, mask):
        """
        Extract all shape features from a single mask.

        Args:
            mask (ndarray): Input mask.

        Returns:
            dict: Dictionary containing shape features.
        """
        single_feature = {
            "circularity": self.get_circularity(mask),
            "solidity": self.get_solidity(mask),
            "eccentricity": self.get_eccentricity(mask)
        }
        return single_feature

    def extract_all_shape_features(self):
        """
        Extract shape features for all masks.

        Returns:
            dict: Dictionary containing feature lists for the whole dataset.
        """
        self.circularity = []
        self.solidity = []
        self.eccentricity = []

        for mask in self.masks:
            self.circularity.append(self.get_circularity(mask))
            self.solidity.append(self.get_solidity(mask))
            self.eccentricity.append(self.get_eccentricity(mask))

        self.shape_feature = {
            "circularity": self.circularity,
            "solidity": self.solidity,
            "eccentricity": self.eccentricity,
            "label": list(self.label)
        }

        return self.shape_feature

In [ ]:
class glcm_extract:

In [ ]:
# These are for exporting the notebook as a PDF later if needed
!apt-get update
!sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc
!jupyter nbconvert --to pdf /content/gdrive/MyDrive/Colab_Notebooks/BMET5933/WEEK_8/Week_8_9_Last_name_Code_file_ipynb_draft_version.ipynb

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pandoc is already the newest version (2.9.2.1-3ubuntu2).
texlive-fonts-recommended is already 